## ▶ Run Online — No Installation Needed

| Platform | Link |
|---|---|
| **Binder** (no account) | [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Piyushjhu/HELIX_Toolbox/main?labpath=examples%2F02_alpss_signal_processing.ipynb) |
| **Google Colab** | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Piyushjhu/HELIX_Toolbox/blob/main/examples/02_alpss_signal_processing.ipynb) |
| **GitHub Codespaces** | [![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/Piyushjhu/HELIX_Toolbox) |

**Using your own data?** The setup cell auto-detects the cloud environment and shows upload instructions.

# Example 2 — ALPSS Signal Processing

This notebook demonstrates the **ALPSS** (Automated Laser PDV Signal Processing)
stage in isolation:

1. Load a raw oscilloscope CSV and inspect the time-domain signal
2. Run ALPSS to extract the velocity trace and uncertainty
3. Visualise the smoothed velocity with uncertainty bands
4. Inspect the noise fraction and IQ detection quality

---
**Before running:** update the path variables in the next cell.

In [ ]:
import os, sys, subprocess

# ── Cloud / Online environment setup ──────────────────────────────────────
try:
    import google.colab
    _ENV = "colab"
except ImportError:
    _ENV = "binder" if os.environ.get("BINDER_SERVICE_HOST") else "local"

if _ENV in ("colab", "binder"):
    os.environ["QT_QPA_PLATFORM"] = "offscreen"
    os.environ["MPLBACKEND"] = "Agg"

if _ENV == "colab":
    REPO_ROOT = "/content/HELIX_Toolbox"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Piyushjhu/HELIX_Toolbox.git",
                        REPO_ROOT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    os.path.join(REPO_ROOT, "requirements.txt")], check=False)
else:
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

SAMPLE_DIR = os.path.join(REPO_ROOT, "input_data", "C1_files")
_sample_files = sorted(f for f in os.listdir(SAMPLE_DIR) if f.endswith(".csv")) if os.path.isdir(SAMPLE_DIR) else []

print(f"Environment: {_ENV} | REPO_ROOT: {REPO_ROOT}")
print(f"Bundled sample files: {_sample_files}")
if _ENV == "colab":
    print("\nTo upload YOUR OWN data:  from google.colab import files; uploaded = files.upload()")
elif _ENV == "binder":
    print("\nTo upload YOUR OWN data: use the Jupyter file-browser (left panel) to upload CSVs.")
# ──────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ── USER PATHS — edit these ────────────────────────────────────────────────
# Default to the first bundled sample file; replace with your own path:
INPUT_FILE   = os.path.join(SAMPLE_DIR, _sample_files[0]) if _sample_files else "/path/to/your/raw_pdv_file.csv"
OUTPUT_DIR   = os.path.join(REPO_ROOT, "examples", "figures", "example_02_output")
HEADER_LINES = 22       # header rows to skip in the oscilloscope CSV
SAMPLE_RATE  = 128e9    # oscilloscope sample rate (S/s)
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nINPUT_FILE : {INPUT_FILE}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print("Ready.")

## 1. Inspect the raw signal

In [ ]:
raw = pd.read_csv(INPUT_FILE, header=None, skiprows=HEADER_LINES, nrows=5000)
time_raw = raw.iloc[:, 0].values
volt_raw = raw.iloc[:, 1].values

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time_raw * 1e9, volt_raw, lw=0.7, color='steelblue')
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Voltage (V)")
ax.set_title("Raw oscilloscope signal (first 5000 samples)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "raw_signal.png"), dpi=150)
plt.show()

## 2. Run ALPSS on the file

We call the CLI with `--analysis-mode alpss_only` so only the ALPSS step runs.

In [ ]:
import subprocess
from helix_analysis_toolbox import save_config_to_file

alpss_config = {
    "save_data":                   "yes",
    "display_plots":               "no",
    "save_all_plots":              "no",
    "header_lines":                HEADER_LINES,
    "time_to_take":                4e-6,
    "use_robust_iq_detection":     True,
    "iq_threshold_factor":         0.8,
    "smoothing_type":              "savgol",
    "smoothing_window_ns":         6.0,
    "savgol_polyorder":            3,
    "use_notch_filter":            False,
    "sample_rate":                 SAMPLE_RATE,
    "save_velocity_smooth_uncert_csv": True,
    "save_noise_csv":              True,
    "C0": 3950.0, "density": 8960.0, "lam": 1.55e-6, "theta": 0.0,
}

cfg = {
    "cli_settings": {
        "input_files":   [INPUT_FILE],
        "input_dir":     None,
        "output_dir":    OUTPUT_DIR,
        "param_folder":  None,
        "analysis_mode": "alpss_only",
        "spade_mode":    "auto",
        "input_pattern": "*.csv",
        "spade_input_files": None,
        "spade_input_dir": None,
        "spade_input_pattern": "*--vel-smooth-with-uncert.csv",
    },
    "alpss_config": alpss_config,
    "spade_config": {},
    "material_properties": {},
}

cfg_path = os.path.join(OUTPUT_DIR, "alpss_only.yml")
save_config_to_file(cfg, cfg_path)

result = subprocess.run(
    [sys.executable, os.path.join(REPO_ROOT, "helix_cli_runner.py"),
     "--config", cfg_path],
    capture_output=False, text=True,
)
print("\nExit code:", result.returncode)

## 3. Load and plot the velocity trace + uncertainty

In [ ]:
import glob as _glob

# Find the smoothed velocity + uncertainty CSV produced by ALPSS
vel_files = _glob.glob(os.path.join(OUTPUT_DIR, "**", "*vel-smooth-with-uncert.csv"), recursive=True)
if not vel_files:
    print("No velocity file found — check the ALPSS run output above.")
else:
    vf = vel_files[0]
    print("Loading:", vf)
    vdf = pd.read_csv(vf, header=0)
    display(vdf.head())

    # Detect column names flexibly
    t_col = vdf.columns[0]
    v_col = vdf.columns[1]
    u_col = vdf.columns[2] if len(vdf.columns) > 2 else None

    t_ns = vdf[t_col].values * 1e9   # convert s → ns
    v    = vdf[v_col].values

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t_ns, v, lw=1.5, color='steelblue', label='Velocity (smoothed)')

    if u_col:
        u = vdf[u_col].values
        ax.fill_between(t_ns, v - u, v + u,
                        alpha=0.25, color='steelblue', label='Uncertainty (±1σ)')

    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Free-surface velocity (m/s)")
    ax.set_title("ALPSS output — smoothed velocity with uncertainty")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "velocity_trace.png"), dpi=150)
    plt.show()

## 4. Key ALPSS parameters explained

| Parameter | What it controls | Typical value |
|-----------|-----------------|---------------|
| `smoothing_window_ns` | Savitzky-Golay window length in ns | 5 – 10 ns |
| `iq_threshold_factor` | Fraction of peak amplitude for IQ onset detection | 0.6 – 0.9 |
| `iq_persistence_ns` | Min duration above threshold to accept onset | 0.3 – 1.0 ns |
| `use_notch_filter` | Remove static carrier frequency | `false` unless carrier is visible in spectrogram |
| `uncert_mult` | Multiplier on raw frequency uncertainty | 5 – 15 |

A higher `smoothing_window_ns` reduces noise but smears sharp velocity
transitions; a lower value retains features but produces noisier uncertainty bands.